# Advanced: Multi-Strategy Uniform Numeric Noise with PAMOLA.CORE

**Goal**: Master all uniform noise strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Additive noise with bounds enforcement
- **Strategy 2**: Multiplicative noise with scaling
- **Strategy 3**: Statistical scaling (scale_by_std)
- **Strategy 4**: Zero preservation with integer rounding
- Calculate advanced privacy metrics
- Analyze noise uniformity and effectiveness
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/noise/
        └── 02_uniform_numeric_noise_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.noise.uniform_numeric_op import (
        UniformNumericNoiseOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, means, std devs)

**Dataset features:**
- 1000 records for comprehensive testing
- Multiple numeric field types
- Realistic sensor data patterns
- Suitable for all noise strategies

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic sensor data with different characteristics
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'temperature': np.round(np.random.normal(22.5, 2.5, n_records), 2),  # °C
        'humidity': np.round(np.random.normal(55, 10, n_records), 2),        # %
        'pressure': np.round(np.random.normal(1013.25, 5, n_records), 2),   # hPa
        'count': np.random.poisson(100, n_records),                           # Integer counts
        'ratio': np.round(np.random.beta(2, 5, n_records), 4)                # 0-1 range
    })
    
    # Add some zero values for testing preserve_zero
    zero_indices = np.random.choice(n_records, size=50, replace=False)
    df.loc[zero_indices, 'count'] = 0
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}, std={df[col].std():.2f}")
        if col == 'count':
            zero_count = (df[col] == 0).sum()
            print(f"    Zero values: {zero_count}")

print(f"\n💡 Perfect dataset for testing all 4 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "years_of_experience"` → YOUR column name
   - `"strategy2_field": "years_of_experience"` → YOUR column name
   - `"strategy3_field": "years_of_experience"` → YOUR column name
   - `"strategy4_field": "years_of_experience"` → YOUR column name
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Data type and range information per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must be numeric and exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "years_of_experience",  # Additive with bounds
    "strategy2_field": "years_of_experience",     # Multiplicative
    "strategy3_field": "years_of_experience",     # Statistical scaling
    "strategy4_field": "years_of_experience"         # Zero preservation + integer
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    # Check if numeric
    if not pd.api.types.is_numeric_dtype(df[field_name]):
        raise ValueError(f"❌ Field '{field_name}' is not numeric!")
    
    print(f"  ✓ {strategy:20s}: '{field_name}'")
    print(f"      Type: {df[field_name].dtype}, Range: [{df[field_name].min():.2f}, {df[field_name].max():.2f}]")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy uniform noise processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Additive Noise with Bounds

**How to use:**
- Review pre-configured parameters
- Run to execute strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_type='additive'`: Adds noise to values
- `noise_range=2.0`: ±2.0 uniform noise
- `output_min=15.0`: Minimum bound (°C)
- `output_max=30.0`: Maximum bound (°C)
- `mode='ENRICH'`: Keeps original + adds noisy field

**Note:** Creates `{field_name}_bounded` with enforced bounds

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: ADDITIVE NOISE WITH BOUNDS")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Additive + bounds",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = UniformNumericNoiseOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_bounded",
    output_format='csv',
    
    # Noise parameters
    noise_range=2.0,
    noise_type='additive',
    
    # Bounds enforcement
    output_min=15.0,
    output_max=30.0,
    preserve_zero=False,
    
    # Statistical parameters
    scale_by_std=False,
    scale_factor=1.0,
    
    # Randomization
    random_seed=None,
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_additive_bounds',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_additive_bounds' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_bounded"
    
    print(f"\n📊 Sample values (first 5):")
    for i in range(min(5, len(result_df_s1))):
        orig = df[field_s1].iloc[i]
        noisy = result_df_s1[output_field_s1].iloc[i]
        noise = noisy - orig
        print(f"  {orig:7.2f} → {noisy:7.2f} (noise: {noise:+6.2f})")
    
    # Check bounds
    at_min = (result_df_s1[output_field_s1] == 15.0).sum()
    at_max = (result_df_s1[output_field_s1] == 30.0).sum()
    print(f"\n📊 Bounds enforcement: {at_min} at min, {at_max} at max")

## STRATEGY 2: Multiplicative Noise

**How to use:**
- Review pre-configured parameters
- Run to execute strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_type='multiplicative'`: Multiplies by (1 + noise)
- `noise_range=0.1`: ±10% relative noise
- `scale_factor=1.0`: No additional scaling
- `mode='ENRICH'`: Keeps original + adds noisy field

**Note:** Creates `{field_name}_multiplicative` with proportional noise

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MULTIPLICATIVE NOISE")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Multiplicative",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = UniformNumericNoiseOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_multiplicative",
    output_format='csv',
    
    # Noise parameters
    noise_range=0.1,  # ±10%
    noise_type='multiplicative',
    
    # Bounds
    output_min=None,
    output_max=None,
    preserve_zero=False,
    
    # Statistical parameters
    scale_by_std=False,
    scale_factor=1.0,
    
    # Randomization
    random_seed=None,
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_multiplicative',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_multiplicative' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_multiplicative"
    
    print(f"\n📊 Sample values (first 5):")
    for i in range(min(5, len(result_df_s2))):
        orig = df[field_s2].iloc[i]
        noisy = result_df_s2[output_field_s2].iloc[i]
        pct_change = ((noisy - orig) / orig * 100) if orig != 0 else 0
        print(f"  {orig:7.2f} → {noisy:7.2f} ({pct_change:+6.1f}%)")

## STRATEGY 3: Statistical Scaling

**How to use:**
- Review pre-configured parameters
- Run to execute strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_type='additive'`: Additive mode
- `noise_range=0.5`: Base range (scaled by std)
- `scale_by_std=True`: Scales noise by field std dev
- `scale_factor=2.0`: Additional multiplier
- `mode='ENRICH'`: Keeps original + adds noisy field

**Note:** Creates `{field_name}_scaled` with adaptive noise magnitude

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: STATISTICAL SCALING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Statistical scaling",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = UniformNumericNoiseOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_scaled",
    output_format='csv',
    
    # Noise parameters
    noise_range=0.5,
    noise_type='additive',
    
    # Bounds
    output_min=None,
    output_max=None,
    preserve_zero=False,
    
    # Statistical parameters
    scale_by_std=True,   # Scale noise by std dev
    scale_factor=2.0,    # Additional 2x multiplier
    
    # Randomization
    random_seed=None,
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_scaled',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_scaled' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_scaled"
    
    print(f"\n📊 Sample values (first 5):")
    for i in range(min(5, len(result_df_s3))):
        orig = df[field_s3].iloc[i]
        noisy = result_df_s3[output_field_s3].iloc[i]
        noise = noisy - orig
        print(f"  {orig:9.2f} → {noisy:9.2f} (noise: {noise:+7.2f})")
    
    # Show adaptive scaling
    field_std = df[field_s3].std()
    effective_range = 0.5 * field_std * 2.0
    print(f"\n📊 Adaptive scaling: ±{effective_range:.2f} (0.5 × {field_std:.2f} std × 2.0 factor)")

## STRATEGY 4: Zero Preservation + Integer Rounding

**How to use:**
- Review pre-configured parameters
- Run to execute strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `noise_type='additive'`: Additive mode
- `noise_range=5.0`: ±5 units
- `preserve_zero=True`: Keeps zeros unchanged
- `round_to_integer=True`: Rounds to nearest integer
- `mode='ENRICH'`: Keeps original + adds noisy field

**Note:** Creates `{field_name}_integer` with preserved zeros and integer values

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 4: ZERO PRESERVATION + INTEGER ROUNDING")
print("=" * 80 + "\n")

tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Zero preservation",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s4 = UniformNumericNoiseOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy4_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy4_field']}_integer",
    output_format='csv',
    
    # Noise parameters
    noise_range=5.0,
    noise_type='additive',
    
    # Bounds
    output_min=0.0,  # Keep non-negative
    output_max=None,
    preserve_zero=True,  # Keep zeros unchanged
    
    # Integer handling
    round_to_integer=True,  # Round to integers
    
    # Statistical parameters
    scale_by_std=False,
    scale_factor=1.0,
    
    # Randomization
    random_seed=None,
    use_secure_random=True,
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 4 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s4 = operation_s4.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy4_integer',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    **kwargs
)

elapsed_s4 = time.time() - start_time
print(f"\n✅ Strategy 4 completed in {elapsed_s4:.2f}s")

# Load results (NEWEST file)
output_files_s4 = sorted(
    list((task_dir / 'strategy4_integer' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s4:
    result_df_s4 = pd.read_csv(output_files_s4[0])
    field_s4 = FIELD_CONFIG["strategy4_field"]
    output_field_s4 = f"{field_s4}_integer"
    
    print(f"\n📊 Sample values (first 10, including zeros):")
    # Find some zero values
    zero_indices = df[df[field_s4] == 0].index[:5]
    non_zero_indices = df[df[field_s4] != 0].index[:5]
    sample_indices = list(zero_indices) + list(non_zero_indices)
    
    for i in sample_indices[:10]:
        orig = df[field_s4].iloc[i]
        noisy = result_df_s4[output_field_s4].iloc[i]
        noise = noisy - orig
        marker = "(zero preserved)" if orig == 0 else ""
        print(f"  {orig:5.0f} → {noisy:5.0f} (noise: {noise:+5.0f}) {marker}")
    
    # Verify zero preservation
    orig_zeros = (df[field_s4] == 0).sum()
    noisy_zeros = (result_df_s4[output_field_s4] == 0).sum()
    print(f"\n📊 Zero preservation: {orig_zeros} original → {noisy_zeros} after noise")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and characteristics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Strategy characteristics summary

**Note:** Fastest strategy isn't always best - consider use case requirements

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Additive + Bounds): {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Multiplicative):    {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Statistical):       {elapsed_s3:6.2f}s")
print(f"  Strategy 4 (Zero + Integer):    {elapsed_s4:6.2f}s")
print(f"  Total:                          {elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4:6.2f}s")

print(f"\n📈 Strategy Characteristics:")
print(f"  Strategy 1: Fixed absolute noise with bounds (good for constrained ranges)")
print(f"  Strategy 2: Proportional noise (preserves relative relationships)")
print(f"  Strategy 3: Adaptive noise magnitude (scales with data variability)")
print(f"  Strategy 4: Discrete values with special handling (count data)")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: SNR, noise stats, uniformity tests, effectiveness
2. **Visualizations**: Distribution and noise pattern charts

**Validate:**
- ✅ SNR > 5 = acceptable utility
- ✅ Noise mean ≈ 0 = unbiased
- ✅ Uniformity tests pass
- ✅ All records processed successfully

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# Review all strategies
strategy_dirs = [
    ('strategy1_additive_bounds', 'Strategy 1: Additive + Bounds'),
    ('strategy2_multiplicative', 'Strategy 2: Multiplicative'),
    ('strategy3_scaled', 'Strategy 3: Statistical Scaling'),
    ('strategy4_integer', 'Strategy 4: Zero + Integer')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. METRICS JSON (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                if 'actual_noise_mean' in metrics:
                    print(f"   Noise Mean:      {metrics['actual_noise_mean']:+.4f}")
                if 'actual_noise_std' in metrics:
                    print(f"   Noise Std:       {metrics['actual_noise_std']:.4f}")
                if 'signal_to_noise_ratio' in metrics:
                    snr = metrics['signal_to_noise_ratio']
                    print(f"   SNR:             {snr:.2f}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. VISUALIZATIONS (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## Step 7: Calculate Privacy Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates comprehensive privacy metrics

**What you'll see:**
- SNR for each strategy
- Correlation coefficients
- Noise uniformity analysis
- Utility preservation scores

**Note:** Requires result_df_s{N} from strategy execution cells

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_comprehensive_metrics(original, noisy):
    """Calculate comprehensive noise metrics."""
    noise = noisy - original
    snr = original.std() / noise.std() if noise.std() > 0 else float('inf')
    corr = original.corr(noisy)
    
    return {
        'snr': snr,
        'correlation': corr,
        'noise_mean': noise.mean(),
        'noise_std': noise.std(),
        'noise_range': (noise.min(), noise.max())
    }

# Check if strategies completed
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    field_s4 = FIELD_CONFIG["strategy4_field"]
    
    print(f"📊 COMPREHENSIVE METRICS:\n")
    
    # Strategy 1
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        metrics_s1 = calculate_comprehensive_metrics(
            df[field_s1], 
            result_df_s1[f"{field_s1}_bounded"]
        )
        print(f"  Strategy 1 (Additive + Bounds):")
        print(f"    SNR:          {metrics_s1['snr']:.2f}")
        print(f"    Correlation:  {metrics_s1['correlation']:.4f}")
        print(f"    Noise Mean:   {metrics_s1['noise_mean']:+.4f}")
        print(f"    Noise Std:    {metrics_s1['noise_std']:.4f}")
    
    # Strategy 2
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        metrics_s2 = calculate_comprehensive_metrics(
            result_df_s1[field_s2], 
            result_df_s2[f"{field_s2}_multiplicative"]
        )
        print(f"\n  Strategy 2 (Multiplicative):")
        print(f"    SNR:          {metrics_s2['snr']:.2f}")
        print(f"    Correlation:  {metrics_s2['correlation']:.4f}")
        print(f"    Noise Mean:   {metrics_s2['noise_mean']:+.4f}")
        print(f"    Noise Std:    {metrics_s2['noise_std']:.4f}")
    
    # Strategy 3
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        metrics_s3 = calculate_comprehensive_metrics(
            result_df_s2[field_s3], 
            result_df_s3[f"{field_s3}_scaled"]
        )
        print(f"\n  Strategy 3 (Statistical):")
        print(f"    SNR:          {metrics_s3['snr']:.2f}")
        print(f"    Correlation:  {metrics_s3['correlation']:.4f}")
        print(f"    Noise Mean:   {metrics_s3['noise_mean']:+.4f}")
        print(f"    Noise Std:    {metrics_s3['noise_std']:.4f}")
    
    # Strategy 4
    if 'result_df_s4' in locals() and result_df_s4 is not None:
        metrics_s4 = calculate_comprehensive_metrics(
            result_df_s3[field_s4], 
            result_df_s4[f"{field_s4}_integer"]
        )
        print(f"\n  Strategy 4 (Zero + Integer):")
        print(f"    SNR:          {metrics_s4['snr']:.2f}")
        print(f"    Correlation:  {metrics_s4['correlation']:.4f}")
        print(f"    Noise Mean:   {metrics_s4['noise_mean']:+.4f}")
        print(f"    Noise Std:    {metrics_s4['noise_std']:.4f}")
        
except NameError:
    print("⚠️  Run Step 3 first!")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports noisy datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (SNR, correlation)

**Output structure:**
```
advanced_output/
├── strategy1_additive_bounds/
│   └── additive_bounds_noisy_data.csv
├── strategy2_multiplicative/
│   └── multiplicative_noisy_data.csv
├── strategy3_scaled/
│   └── scaled_noisy_data.csv
└── strategy4_integer/
    └── integer_noisy_data.csv
```

**Note:** Files include both original and noisy fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    field_s4 = FIELD_CONFIG["strategy4_field"]
except NameError:
    print("❌ Run Step 3 first!")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Additive + Bounds
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: ADDITIVE + BOUNDS")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_additive_bounds'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_bounded"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['record_id', field_s1, output_col_s1]
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'additive_bounds_noisy_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        # Statistics
        noise_s1 = result_df_s1[output_col_s1] - df[field_s1]
        snr_s1 = df[field_s1].std() / noise_s1.std() if noise_s1.std() > 0 else float('inf')
        corr_s1 = df[field_s1].corr(result_df_s1[output_col_s1])
        print(f"\n📈 Statistics: SNR={snr_s1:.2f}, Correlation={corr_s1:.4f}")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Multiplicative
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: MULTIPLICATIVE")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_multiplicative'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_bounded"
    output_col_s2 = f"{field_s2}_multiplicative"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['record_id', field_s1, output_col_s1, 
                          field_s2, output_col_s2]
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'multiplicative_noisy_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
        
        # Statistics
        noise_s2 = result_df_s2[output_col_s2] - result_df_s1[field_s2]
        snr_s2 = result_df_s1[field_s2].std() / noise_s2.std() if noise_s2.std() > 0 else float('inf')
        corr_s2 = result_df_s1[field_s2].corr(result_df_s2[output_col_s2])
        print(f"\n📈 Statistics: SNR={snr_s2:.2f}, Correlation={corr_s2:.4f}")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Statistical Scaling
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: STATISTICAL SCALING")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_scaled'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_bounded"
    output_col_s2 = f"{field_s2}_multiplicative"
    output_col_s3 = f"{field_s3}_scaled"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['record_id', field_s1, output_col_s1,
                          field_s2, output_col_s2, field_s3, output_col_s3]
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'scaled_noisy_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
        
        # Statistics
        noise_s3 = result_df_s3[output_col_s3] - result_df_s2[field_s3]
        snr_s3 = result_df_s2[field_s3].std() / noise_s3.std() if noise_s3.std() > 0 else float('inf')
        corr_s3 = result_df_s2[field_s3].corr(result_df_s3[output_col_s3])
        print(f"\n📈 Statistics: SNR={snr_s3:.2f}, Correlation={corr_s3:.4f}")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# STRATEGY 4: Zero + Integer
# ============================================================================
if 'result_df_s4' in locals() and result_df_s4 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 4: ZERO + INTEGER")
    print("=" * 80)
    
    s4_dir = export_base_dir / 'strategy4_integer'
    s4_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_bounded"
    output_col_s2 = f"{field_s2}_multiplicative"
    output_col_s3 = f"{field_s3}_scaled"
    output_col_s4 = f"{field_s4}_integer"
    
    if output_col_s4 in result_df_s4.columns:
        export_cols_s4 = ['record_id', field_s1, output_col_s1,
                          field_s2, output_col_s2, field_s3, output_col_s3,
                          field_s4, output_col_s4]
        existing_cols_s4 = [col for col in export_cols_s4 if col in result_df_s4.columns]
        
        final_df_s4 = result_df_s4[existing_cols_s4].copy()
        
        output_path_s4 = s4_dir / 'integer_noisy_data.csv'
        try:
            final_df_s4.to_csv(output_path_s4, index=False)
            print(f"\n✅ Saved: {output_path_s4}")
            print(f"   Rows: {len(final_df_s4):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s4.head())
        
        # Statistics
        noise_s4 = result_df_s4[output_col_s4] - result_df_s3[field_s4]
        snr_s4 = result_df_s3[field_s4].std() / noise_s4.std() if noise_s4.std() > 0 else float('inf')
        corr_s4 = result_df_s3[field_s4].corr(result_df_s4[output_col_s4])
        print(f"\n📈 Statistics: SNR={snr_s4:.2f}, Correlation={corr_s4:.4f}")
        
        # Zero preservation check
        orig_zeros = (df[field_s4] == 0).sum()
        noisy_zeros = (result_df_s4[output_col_s4] == 0).sum()
        print(f"   Zeros: {orig_zeros} → {noisy_zeros} (preserved: {orig_zeros == noisy_zeros})")
    else:
        print(f"\n❌ Column '{output_col_s4}' not found")
else:
    print("\n⚠️  Strategy 4 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 4 noise strategies implemented and compared
- ✅ Privacy metrics calculated (SNR, correlation, uniformity)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Additive + Bounds**: Fixed noise with range constraints
- **Multiplicative**: Proportional noise preserving relative relationships
- **Statistical Scaling**: Adaptive noise based on data variability
- **Zero + Integer**: Special handling for count data and discrete values

**Strategy recommendations:**
- **Use Additive + Bounds** when: Values must stay within physical limits
- **Use Multiplicative** when: Relative differences more important than absolute
- **Use Statistical Scaling** when: Data has varying scales or high variability
- **Use Zero + Integer** when: Working with counts or discrete measurements

**Privacy-Utility Guidelines:**
- **SNR > 10**: High utility, moderate privacy (safe for most use cases)
- **SNR 5-10**: Balanced privacy-utility (recommended for sensitive data)
- **SNR < 5**: Strong privacy, lower utility (high-risk scenarios)
- **Correlation > 0.9**: Excellent data preservation
- **Correlation 0.7-0.9**: Good preservation for most analytics

**Next steps:**
- Test with your own datasets
- Tune noise parameters for optimal SNR
- Deploy to production environment
- Consider combining with other privacy techniques

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)